In [3]:
import os
import cv2
import pandas as pd

# ============================================================
# Select Dataset for train and augemnted change path manually
# ============================================================

dataset = input("Which dataset do you want to label? (train / valid / test): ").strip().lower()

BASE_FOLDER = r"../data_set"

if dataset == "train":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "train")
    CSV_FILE = os.path.join(BASE_FOLDER, "train.csv")

elif dataset in ["valid", "validation", "val"]:
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "valid")
    CSV_FILE = os.path.join(BASE_FOLDER, "valid.csv")

elif dataset == "test":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "test")
    CSV_FILE = os.path.join(BASE_FOLDER, "test.csv")

else:
    print("Invalid dataset.")
    exit()

SUPPORTED_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

# ============================================================
# Resume Support
# ============================================================

if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE, dtype={"label": str})
    labeled_images = set(df["image"])
else:
    df = pd.DataFrame(columns=["image", "label"])
    labeled_images = set()

# ============================================================
# Image List
# ============================================================

images = sorted([
    img for img in os.listdir(IMAGE_FOLDER)
    if img.lower().endswith(SUPPORTED_EXTENSIONS)
])

print(f"\nDataset : {dataset}")
print(f"Total Images found in folder : {len(images)}")
print(f"Already Labeled in CSV       : {len(labeled_images)}")

# ============================================================
# Helper function to get base name
# ============================================================
def get_original_stem(filename):
    """ Extracts base file string before '_crop_X' or '_no_detection' """
    stem = os.path.splitext(filename)[0]
    if "_no_detection" in stem:
        return stem.split("_no_detection")[0]
    if "_crop_" in stem:
        return stem.split("_crop_")[0]
    return stem

# ============================================================
# Labeling Loop
# ============================================================

for index, image_name in enumerate(images):

    # CRITICAL FIX: If this exact image or its root crop origin is already in the CSV, skip it!
    base_stem = get_original_stem(image_name)
    
    # Matches exact name OR checks if the base name exists in any recorded rows
    if image_name in labeled_images or any(base_stem in labeled for labeled in labeled_images):
        continue

    image_path = os.path.join(IMAGE_FOLDER, image_name)

    img = cv2.imread(image_path)

    if img is None:
        print(f"Could not read {image_name}")
        continue

    # Resize only for display
    h, w = img.shape[:2]
    scale = min(900 / w, 400 / h)
    display = cv2.resize(img, (int(w * scale), int(h * scale)))
    cv2.namedWindow("Meter Labeling Tool", cv2.WINDOW_NORMAL)
    cv2.imshow("Meter Labeling Tool", display)
    cv2.resizeWindow("Meter Labeling Tool", 900, 400)

    # Force the image to appear before asking for input
    cv2.waitKey(100)

    print(f"\n[{index+1}/{len(images)}] {image_name}")
    while True:

        label = input("Enter 5-digit reading (q = quit, s = skip, d = delete): ").strip()

        if label.lower() == "q":
            cv2.destroyAllWindows()
            df.to_csv(CSV_FILE, index=False)
            print("\nProgress Saved.")
            exit()

        if label.lower() == "s":
            break

        # -----------------------------
        # NEW FEATURE: Delete Image
        # -----------------------------
        if label.lower() in ["d", "del", "delete"]:
            cv2.destroyWindow("Meter Labeling Tool") # Close window to avoid OS file locks
            try:
                os.remove(image_path)
                print(f"🔥 Deleted file: {image_name}")
            except Exception as e:
                print(f"Error deleting file: {e}")
            break

        if len(label) == 5 and label.isdigit():
            df.loc[len(df)] = [image_name, label]
            df.to_csv(CSV_FILE, index=False)
            break

        print("Invalid label! Please enter exactly 5 digits.")

cv2.destroyAllWindows()

print("\n===================================")
print("All pending images have been processed!")
print("CSV saved to:")
print(CSV_FILE)
print("===================================")


Dataset : test
Total Images found in folder : 323
Already Labeled in CSV       : 25

[21/323] 0642_10216468990_crop_0.png

[34/323] 0642_13508310003_crop_0.png

[44/323] 74059099593_crop_0.png
🔥 Deleted file: 74059099593_crop_0.png

[45/323] 74291476724_crop_0.png
🔥 Deleted file: 74291476724_crop_0.png

[46/323] 74345568815_crop_0.png
🔥 Deleted file: 74345568815_crop_0.png

[47/323] 74378703248_crop_0.png

[48/323] 74460590297_crop_0.png

[49/323] 74507910003_crop_0.png

[50/323] 74508091670_crop_0.png

[51/323] 74519610005_crop_0.png

[52/323] 74546215067_crop_0.png
🔥 Deleted file: 74546215067_crop_0.png

[53/323] 74582410002_crop_0.png
🔥 Deleted file: 74582410002_crop_0.png

[54/323] 74584225242_crop_0.png

[55/323] 74590510009_crop_0.png
🔥 Deleted file: 74590510009_crop_0.png

[56/323] 74683849983_crop_0.png

[57/323] 74703100003_crop_0.png

[58/323] 74760298393_crop_0.png

[59/323] 75008220008_crop_0.png

[60/323] 75112211307_crop_0.png
🔥 Deleted file: 75112211307_crop_0.png

[61/

In [3]:
import pandas as pd

df = pd.read_csv("../data_set/train.csv")

print(df.head(50))
print(df.columns)

                         image  label
0       0642_00409673340_0.png     19
1       0642_00897310009_0.png     15
2       0642_01608310007_0.png     16
3       0642_01997310006_0.png      3
4       0642_02001960166_0.png     11
5       0642_02608310005_0.png      9
6       0642_02708310004_0.png      6
7       0642_03508310004_0.png      7
8       0642_03903021420_0.png  10421
9       0642_15014143919_0.png   5861
10      0642_15208310001_0.png    871
11      0642_15300714548_0.png    120
12      0642_15737888402_0.png    487
13      0642_16319916454_0.png     14
14      0642_17408310005_0.png     26
15  0642_17408310005_233_0.png      4
16      0642_18208310005_0.png    336
17      0642_18697310003_0.png   3822
18      0642_18698440072_0.png    115
19      0642_19226508661_0.png     10
20      0642_19307756163_0.png     11
21      0642_20790083834_0.png     30
22      0642_20971218530_0.png    124
23      0642_21987928039_0.png    212
24      0642_22189042033_0.png     19
25      0642